In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


SEED = 42
set_seed(SEED)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# zip_path = "/content/drive/MyDrive/CSCI 5527 Project/Dataset Preprocessing/processed_eeg_dataset.zip"
# extract_path = "/content/drive/MyDrive/CSCI 5527 Project/Dataset Preprocessing/processed_eeg_dataset"

# !mkdir -p "$extract_path"
# !unzip -q "$zip_path" -d "$extract_path"

In [ ]:
DATA_DIR = Path("/content/drive/MyDrive/CSCI 5527 Project/Dataset Preprocessing/processed_eeg_dataset/processed_eeg_dataset")
SPLIT_DIR = Path("splits")
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15

assert abs(TRAIN_SIZE + VAL_SIZE + TEST_SIZE - 1.0) < 1e-8

In [ ]:
metadata_path = DATA_DIR / "eeg_metadata.csv"
labels_path = DATA_DIR / "y_labels.npy"
groups_path = DATA_DIR / "groups.npy"
images_path = DATA_DIR / "X_images.npy"

metadata = pd.read_csv(metadata_path)
y = np.load(labels_path, allow_pickle=True)
groups = np.load(groups_path, allow_pickle=True)

print("metadata shape:", metadata.shape)
print("y shape       :", y.shape)
print("groups shape  :", groups.shape)

metadata shape: (8300, 13)
y shape       : (8300,)
groups shape  : (8300,)


In [ ]:
def verify_alignment(metadata, y, groups):
    n_meta = len(metadata)
    n_y = len(y)
    n_groups = len(groups)

    if not (n_meta == n_y == n_groups):
        raise ValueError(
            f"Length mismatch: metadata={n_meta}, y={n_y}, groups={n_groups}"
        )

    required_cols = {"subject_id", "session", "label", "file_path", "epoch_index"}
    missing = required_cols - set(metadata.columns)
    if missing:
        raise ValueError(f"Metadata missing required columns: {missing}")

    if not np.array_equal(metadata["label"].to_numpy(), y):
        raise ValueError("Mismatch between metadata['label'] and y_labels.npy")

    if not np.array_equal(metadata["subject_id"].astype(str).to_numpy(), groups.astype(str)):
        raise ValueError("Mismatch between metadata['subject_id'] and groups.npy")

    print("Alignment check passed.")


verify_alignment(metadata, y, groups)

Alignment check passed.


In [ ]:
def make_group_splits(y, groups, train_size=0.70, val_size=0.15, test_size=0.15, seed=42):
    assert abs(train_size + val_size + test_size - 1.0) < 1e-8

    n_samples = len(y)
    all_indices = np.arange(n_samples)

    # First split: train vs temp
    gss_1 = GroupShuffleSplit(
        n_splits=1,
        train_size=train_size,
        random_state=seed
    )
    train_idx, temp_idx = next(gss_1.split(all_indices, y, groups))

    # Second split: val vs test from temp
    temp_y = y[temp_idx]
    temp_groups = groups[temp_idx]

    relative_val = val_size / (val_size + test_size)

    gss_2 = GroupShuffleSplit(
        n_splits=1,
        train_size=relative_val,
        random_state=seed + 1
    )
    temp_train_idx, temp_test_idx = next(
        gss_2.split(np.arange(len(temp_idx)), temp_y, temp_groups)
    )

    val_idx = temp_idx[temp_train_idx]
    test_idx = temp_idx[temp_test_idx]

    return {
        "train": np.sort(train_idx),
        "val": np.sort(val_idx),
        "test": np.sort(test_idx),
    }


splits = make_group_splits(
    y=y,
    groups=groups,
    train_size=TRAIN_SIZE,
    val_size=VAL_SIZE,
    test_size=TEST_SIZE,
    seed=SEED,
)

In [ ]:
def assert_no_group_overlap(groups, splits):
    train_groups = set(groups[splits["train"]].astype(str))
    val_groups = set(groups[splits["val"]].astype(str))
    test_groups = set(groups[splits["test"]].astype(str))

    assert len(train_groups & val_groups) == 0, "Subject overlap between train and val"
    assert len(train_groups & test_groups) == 0, "Subject overlap between train and test"
    assert len(val_groups & test_groups) == 0, "Subject overlap between val and test"

    print("No subject leakage across splits.")


assert_no_group_overlap(groups, splits)

No subject leakage across splits.


In [ ]:
def summarize_split(name, indices, metadata, y, groups):
    split_meta = metadata.iloc[indices].copy()

    print(f"\n===== {name.upper()} SPLIT =====")
    print("num samples   :", len(indices))
    print("num subjects  :", split_meta["subject_id"].nunique())
    print("num recordings:", split_meta["file_path"].nunique())
    print("label counts  :", pd.Series(y[indices]).value_counts().sort_index().to_dict())
    print("session counts:", split_meta["session"].value_counts().to_dict())


for split_name, idx in splits.items():
    summarize_split(split_name, idx, metadata, y, groups)


===== TRAIN SPLIT =====
num samples   : 5738
num subjects  : 49
num recordings: 96
label counts  : {0: 2940, 1: 2798}
session counts: {'ses-1': 2940, 'ses-2': 2798}

===== VAL SPLIT =====
num samples   : 1246
num subjects  : 11
num recordings: 21
label counts  : {0: 649, 1: 597}
session counts: {'ses-1': 649, 'ses-2': 597}

===== TEST SPLIT =====
num samples   : 1316
num subjects  : 11
num recordings: 22
label counts  : {0: 660, 1: 656}
session counts: {'ses-1': 660, 'ses-2': 656}


In [ ]:
# Save split index arrays
for split_name, idx in splits.items():
    np.save(SPLIT_DIR / f"{split_name}_indices.npy", idx)

# Save split metadata CSVs
for split_name, idx in splits.items():
    split_df = metadata.iloc[idx].copy()
    split_df["global_index"] = idx
    split_df.to_csv(SPLIT_DIR / f"{split_name}_metadata.csv", index=False)

# Save one master assignment CSV
all_assignments = metadata.copy()
all_assignments["global_index"] = np.arange(len(metadata))
all_assignments["split"] = "UNASSIGNED"

for split_name, idx in splits.items():
    all_assignments.loc[idx, "split"] = split_name

all_assignments.to_csv(SPLIT_DIR / "all_split_assignments.csv", index=False)

# Save summary json
summary = {
    "seed": SEED,
    "train_size": TRAIN_SIZE,
    "val_size": VAL_SIZE,
    "test_size": TEST_SIZE,
    "overall_num_samples": int(len(metadata)),
    "overall_num_subjects": int(metadata["subject_id"].nunique()),
    "overall_label_counts": {
        str(k): int(v) for k, v in pd.Series(y).value_counts().sort_index().to_dict().items()
    },
    "splits": {}
}

for split_name, idx in splits.items():
    split_meta = metadata.iloc[idx]
    summary["splits"][split_name] = {
        "num_samples": int(len(idx)),
        "num_subjects": int(split_meta["subject_id"].nunique()),
        "num_recordings": int(split_meta["file_path"].nunique()),
        "label_counts": {
            str(k): int(v) for k, v in pd.Series(y[idx]).value_counts().sort_index().to_dict().items()
        },
        "session_counts": {
            str(k): int(v) for k, v in split_meta["session"].value_counts().to_dict().items()
        },
    }

with open(SPLIT_DIR / "split_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Split artifacts saved to:", SPLIT_DIR)

Split artifacts saved to: splits


In [ ]:
class EEGImageDataset(Dataset):
    def __init__(
        self,
        data_dir,
        split_indices_path,
        metadata_csv_path=None,
        transform=None,
        return_metadata=False,
        mmap_images=True,
    ):
        self.data_dir = Path(data_dir)
        self.split_indices_path = Path(split_indices_path)
        self.transform = transform
        self.return_metadata = return_metadata

        if metadata_csv_path is None:
            metadata_csv_path = self.data_dir / "eeg_metadata.csv"
        else:
            metadata_csv_path = Path(metadata_csv_path)

        x_images_path = self.data_dir / "X_images.npy"
        y_labels_path = self.data_dir / "y_labels.npy"
        groups_path = self.data_dir / "groups.npy"

        if not x_images_path.exists():
            raise FileNotFoundError(f"Missing file: {x_images_path}")
        if not y_labels_path.exists():
            raise FileNotFoundError(f"Missing file: {y_labels_path}")
        if not groups_path.exists():
            raise FileNotFoundError(f"Missing file: {groups_path}")
        if not metadata_csv_path.exists():
            raise FileNotFoundError(f"Missing file: {metadata_csv_path}")
        if not self.split_indices_path.exists():
            raise FileNotFoundError(f"Missing file: {self.split_indices_path}")

        mmap_mode = "r" if mmap_images else None
        self.x_images = np.load(x_images_path, mmap_mode=mmap_mode, allow_pickle=False)
        self.y = np.load(y_labels_path, allow_pickle=True)
        self.groups = np.load(groups_path, allow_pickle=True)
        self.metadata = pd.read_csv(metadata_csv_path)
        self.indices = np.load(self.split_indices_path)

        self._verify()

    def _verify(self):
        n_total = len(self.metadata)

        if len(self.x_images) != n_total:
            raise ValueError(f"X_images length mismatch: {len(self.x_images)} vs metadata {n_total}")
        if len(self.y) != n_total:
            raise ValueError(f"y length mismatch: {len(self.y)} vs metadata {n_total}")
        if len(self.groups) != n_total:
            raise ValueError(f"groups length mismatch: {len(self.groups)} vs metadata {n_total}")

        if np.any(self.indices < 0) or np.any(self.indices >= n_total):
            raise ValueError("Split indices contain out-of-range values")

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        global_idx = int(self.indices[idx])

        image = self.x_images[global_idx]   # shape: (3, H, W)
        label = int(self.y[global_idx])
        group = str(self.groups[global_idx])

        image = torch.from_numpy(np.asarray(image)).float() / 255.0
        label = torch.tensor(label, dtype=torch.long)

        if self.transform is not None:
            image = self.transform(image)

        if not self.return_metadata:
            return image, label

        row = self.metadata.iloc[global_idx].to_dict()
        meta = {
            "global_index": global_idx,
            "group": group,
            **row,
        }
        return image, label, meta

    def get_split_labels(self):
        return self.y[self.indices]

    def get_split_groups(self):
        return self.groups[self.indices]

In [ ]:
train_dataset = EEGImageDataset(
    data_dir=DATA_DIR,
    split_indices_path=SPLIT_DIR / "train_indices.npy",
    return_metadata=True,
)

val_dataset = EEGImageDataset(
    data_dir=DATA_DIR,
    split_indices_path=SPLIT_DIR / "val_indices.npy",
    return_metadata=True,
)

test_dataset = EEGImageDataset(
    data_dir=DATA_DIR,
    split_indices_path=SPLIT_DIR / "test_indices.npy",
    return_metadata=True,
)

print("Train size:", len(train_dataset))
print("Val size  :", len(val_dataset))
print("Test size :", len(test_dataset))

Train size: 5738
Val size  : 1246
Test size : 1316


In [ ]:
image, label, meta = train_dataset[0]

print("Image shape :", image.shape)
print("Label       :", label.item())
print("Subject     :", meta["subject_id"])
print("Session     :", meta["session"])
print("Epoch index :", meta["epoch_index"])
print("File path   :", meta["file_path"])
print("Global index:", meta["global_index"])

Image shape : torch.Size([3, 61, 1250])
Label       : 0
Subject     : sub-02
Session     : ses-1
Epoch index : 0
File path   : data\sub-02\ses-1\eeg\sub-02_ses-1_task-eyesopen_eeg.set
Global index: 60


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("DataLoaders ready.")

DataLoaders ready.


# Model

In [ ]:
!pip -q install timm

In [ ]:
import timm
import torch.nn as nn
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode

In [ ]:
MODEL_NAME = "deit_tiny_patch16_224"
IMAGE_SIZE = 224
NUM_CLASSES = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Model      :", MODEL_NAME)
print("Image size :", IMAGE_SIZE)
print("Device     :", DEVICE)

Model      : deit_tiny_patch16_224
Image size : 224
Device     : cuda


In [ ]:
vit_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BILINEAR, antialias=True),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

In [ ]:
train_dataset = EEGImageDataset(
    data_dir=DATA_DIR,
    split_indices_path=SPLIT_DIR / "train_indices.npy",
    transform=vit_transform,
    return_metadata=True,
)

val_dataset = EEGImageDataset(
    data_dir=DATA_DIR,
    split_indices_path=SPLIT_DIR / "val_indices.npy",
    transform=vit_transform,
    return_metadata=True,
)

test_dataset = EEGImageDataset(
    data_dir=DATA_DIR,
    split_indices_path=SPLIT_DIR / "test_indices.npy",
    transform=vit_transform,
    return_metadata=True,
)

print("Train size:", len(train_dataset))
print("Val size  :", len(val_dataset))
print("Test size :", len(test_dataset))

Train size: 5738
Val size  : 1246
Test size : 1316


In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("Loaders ready.")

Loaders ready.


In [ ]:
image, label, meta = train_dataset[0]

print("Image shape after transform:", image.shape)
print("Label                      :", label.item())
print("Subject                    :", meta["subject_id"])
print("Session                    :", meta["session"])

Image shape after transform: torch.Size([3, 224, 224])
Label                      : 0
Subject                    : sub-02
Session                    : ses-1


In [ ]:
model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=NUM_CLASSES,
)

model = model.to(DEVICE)
print(model.__class__.__name__)

VisionTransformer


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

Total params    : 5,524,802
Trainable params: 5,524,802


In [ ]:
batch = next(iter(train_loader))
images, labels, metas = batch

print("Batch image shape :", images.shape)
print("Batch labels shape:", labels.shape)

images = images.to(DEVICE)
labels = labels.to(DEVICE)

with torch.no_grad():
    logits = model(images)

print("Logits shape:", logits.shape)

Batch image shape : torch.Size([32, 3, 224, 224])
Batch labels shape: torch.Size([32])
Logits shape: torch.Size([32, 2])


In [ ]:
print(type(metas))
print(metas.keys())
print("First few subjects:", metas["subject_id"][:5])
print("First few sessions:", metas["session"][:5])

<class 'dict'>
dict_keys(['global_index', 'group', 'subject_id', 'session', 'task', 'label', 'label_name', 'epoch_index', 'sfreq', 'n_channels', 'n_times', 'img_channels', 'img_height', 'img_width', 'file_path'])
First few subjects: ['sub-71', 'sub-57', 'sub-37', 'sub-03', 'sub-15']
First few sessions: ['ses-1', 'ses-1', 'ses-2', 'ses-1', 'ses-1']


# Experiments

In [ ]:
import copy
import json
import time
from pathlib import Path

from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

import torch
import torch.nn as nn
from torch.optim import AdamW

In [ ]:
EXPERIMENT_DIR = Path("vit_experiment")
CHECKPOINT_DIR = EXPERIMENT_DIR / "checkpoints"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 20
PATIENCE = 5
MONITOR_METRIC = "balanced_accuracy"   # validation metric used for model selection

run_config = {
    "model_name": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "num_classes": NUM_CLASSES,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "num_epochs": NUM_EPOCHS,
    "patience": PATIENCE,
    "monitor_metric": MONITOR_METRIC,
    "seed": SEED,
}

with open(EXPERIMENT_DIR / "run_config.json", "w") as f:
    json.dump(run_config, f, indent=2)

print(json.dumps(run_config, indent=2))

{
  "model_name": "deit_tiny_patch16_224",
  "image_size": 224,
  "num_classes": 2,
  "batch_size": 32,
  "lr": 0.0001,
  "weight_decay": 0.0001,
  "num_epochs": 20,
  "patience": 5,
  "monitor_metric": "balanced_accuracy",
  "seed": 42
}


In [ ]:
train_labels = train_dataset.get_split_labels()
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)

class_weights = len(train_labels) / (NUM_CLASSES * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

print("Train class counts:", class_counts)
print("Class weights     :", class_weights)

Train class counts: [2940 2798]
Class weights     : tensor([0.9759, 1.0254], device='cuda:0')


In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, average="binary"),
    }

    try:
        metrics["auroc"] = roc_auc_score(y_true, y_prob)
    except ValueError:
        metrics["auroc"] = float("nan")

    return metrics

In [ ]:
def save_checkpoint(path, model, optimizer, epoch, metric_value, history):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_metric": metric_value,
        "history": history,
    }
    torch.save(checkpoint, path)


def load_checkpoint(path, model, optimizer=None, map_location="cpu"):
    checkpoint = torch.load(path, map_location=map_location)

    model.load_state_dict(checkpoint["model_state_dict"])

    if optimizer is not None and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    return checkpoint

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, epoch=None, num_epochs=None):
    model.train()

    running_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []

    progress = tqdm(loader, desc=f"Train {epoch}/{num_epochs}", leave=False)

    for images, labels, _ in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = torch.argmax(logits, dim=1)

        all_labels.extend(labels.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())
        all_probs.extend(probs.detach().cpu().numpy())

        current_loss = running_loss / len(all_labels)
        progress.set_postfix(loss=f"{current_loss:.4f}")

    epoch_loss = running_loss / len(loader.dataset)
    metrics = compute_metrics(
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_probs),
    )

    return epoch_loss, metrics

In [ ]:
def evaluate_one_epoch(model, loader, criterion, device, split_name="Val", epoch=None, num_epochs=None):
    model.eval()

    running_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []
    all_global_indices = []

    progress = tqdm(loader, desc=f"{split_name} {epoch}/{num_epochs}", leave=False)

    with torch.no_grad():
        for images, labels, metas in progress:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, labels)

            running_loss += loss.item() * images.size(0)

            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = torch.argmax(logits, dim=1)

            all_labels.extend(labels.detach().cpu().numpy())
            all_preds.extend(preds.detach().cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

            # metas is a dict of lists after DataLoader collation
            if "global_index" in metas:
                all_global_indices.extend(metas["global_index"])

            current_loss = running_loss / len(all_labels)
            progress.set_postfix(loss=f"{current_loss:.4f}")

    epoch_loss = running_loss / len(loader.dataset)
    metrics = compute_metrics(
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_probs),
    )

    outputs = {
        "loss": epoch_loss,
        "metrics": metrics,
        "y_true": np.array(all_labels),
        "y_pred": np.array(all_preds),
        "y_prob": np.array(all_probs),
        "global_index": np.array(all_global_indices),
    }
    return outputs

In [ ]:
history = []

best_metric = -np.inf
best_epoch = -1
epochs_without_improvement = 0

best_ckpt_path = CHECKPOINT_DIR / "best_checkpoint.pt"
last_ckpt_path = CHECKPOINT_DIR / "last_checkpoint.pt"

start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_metrics = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE,
        epoch=epoch,
        num_epochs=NUM_EPOCHS,
    )

    val_outputs = evaluate_one_epoch(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=DEVICE,
        split_name="Val",
        epoch=epoch,
        num_epochs=NUM_EPOCHS,
    )

    val_loss = val_outputs["loss"]
    val_metrics = val_outputs["metrics"]

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    history.append(row)

    history_df = pd.DataFrame(history)
    history_df.to_csv(EXPERIMENT_DIR / "training_history.csv", index=False)

    current_metric = val_metrics[MONITOR_METRIC]
    improved = current_metric > best_metric

    # Always save last checkpoint
    save_checkpoint(
        path=last_ckpt_path,
        model=model,
        optimizer=optimizer,
        epoch=epoch,
        metric_value=current_metric,
        history=history,
    )

    if improved:
        best_metric = current_metric
        best_epoch = epoch
        epochs_without_improvement = 0

        save_checkpoint(
            path=best_ckpt_path,
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            metric_value=current_metric,
            history=history,
        )

        val_pred_df = pd.DataFrame({
            "global_index": val_outputs["global_index"],
            "y_true": val_outputs["y_true"],
            "y_pred": val_outputs["y_pred"],
            "y_prob_sd": val_outputs["y_prob"],
        })
        val_pred_df.to_csv(EXPERIMENT_DIR / "best_val_predictions.csv", index=False)

        status = "improved"
    else:
        epochs_without_improvement += 1
        status = f"no_improve({epochs_without_improvement}/{PATIENCE})"

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"train_bal_acc={train_metrics['balanced_accuracy']:.4f} | "
        f"val_bal_acc={val_metrics['balanced_accuracy']:.4f} | "
        f"val_auroc={val_metrics['auroc']:.4f} | "
        f"{status}"
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}.")
        break

elapsed = time.time() - start_time

summary = {
    "best_epoch": best_epoch,
    "best_val_metric": float(best_metric),
    "monitor_metric": MONITOR_METRIC,
    "elapsed_minutes": elapsed / 60.0,
}

with open(EXPERIMENT_DIR / "training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"\nTraining complete in {elapsed/60:.2f} minutes.")
print(f"Best epoch: {best_epoch}")
print(f"Best val {MONITOR_METRIC}: {best_metric:.4f}")
print(f"Best checkpoint: {best_ckpt_path}")
print(f"Last checkpoint: {last_ckpt_path}")

Train 1/20:   0%|          | 0/180 [00:00<?, ?it/s]

Val 1/20:   0%|          | 0/39 [00:00<?, ?it/s]

Epoch 01/20 | train_loss=0.5756 | val_loss=0.6517 | train_bal_acc=0.6782 | val_bal_acc=0.6636 | val_auroc=0.7038 | improved


Train 2/20:   0%|          | 0/180 [00:00<?, ?it/s]

Val 2/20:   0%|          | 0/39 [00:00<?, ?it/s]

...

Val 7/20:   0%|          | 0/39 [00:00<?, ?it/s]

Epoch 07/20 | train_loss=0.0839 | val_loss=1.4712 | train_bal_acc=0.9687 | val_bal_acc=0.6939 | val_auroc=0.7277 | no_improve(4/5)


Train 8/20:   0%|          | 0/180 [00:00<?, ?it/s]

Val 8/20:   0%|          | 0/39 [00:00<?, ?it/s]

Epoch 08/20 | train_loss=0.0553 | val_loss=1.6692 | train_bal_acc=0.9810 | val_bal_acc=0.7033 | val_auroc=0.7371 | no_improve(5/5)
Early stopping triggered at epoch 8.

Training complete in 1.90 minutes.
Best epoch: 3
Best val balanced_accuracy: 0.7050
Best checkpoint: vit_experiment/checkpoints/best_checkpoint.pt
Last checkpoint: vit_experiment/checkpoints/last_checkpoint.pt


In [ ]:
checkpoint = load_checkpoint(
    path=CHECKPOINT_DIR / "best_checkpoint.pt",
    model=model,
    optimizer=None,
    map_location=DEVICE,
)

model = model.to(DEVICE)
model.eval()

print("Loaded best checkpoint from:", CHECKPOINT_DIR / "best_checkpoint.pt")
print("Checkpoint epoch           :", checkpoint["epoch"])
print("Checkpoint best metric     :", checkpoint["best_metric"])

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 

In [ ]:
val_outputs = evaluate_one_epoch(
    model=model,
    loader=val_loader,
    criterion=criterion,
    device=DEVICE,
    split_name="Val",
)

print("Validation metrics:")
for k, v in val_outputs["metrics"].items():
    print(f"{k}: {v:.4f}")

print("\nValidation confusion matrix:")
print(confusion_matrix(val_outputs["y_true"], val_outputs["y_pred"]))

In [ ]:
test_outputs = evaluate_one_epoch(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=DEVICE,
    split_name="Test",
)

print("Test metrics:")
for k, v in test_outputs["metrics"].items():
    print(f"{k}: {v:.4f}")

print("\nTest confusion matrix:")
print(confusion_matrix(test_outputs["y_true"], test_outputs["y_pred"]))

In [ ]:
test_results_df = pd.DataFrame({
    "global_index": test_outputs["global_index"],
    "y_true": test_outputs["y_true"],
    "y_pred": test_outputs["y_pred"],
    "y_prob_sd": test_outputs["y_prob"],
})

# Join back metadata using global index
meta_lookup = test_dataset.metadata.copy()
meta_lookup["global_index"] = np.arange(len(meta_lookup))

test_results_df = test_results_df.merge(
    meta_lookup,
    on="global_index",
    how="left",
)

test_results_df.to_csv(EXPERIMENT_DIR / "test_predictions.csv", index=False)
test_results_df.head()

In [ ]:
history_df = pd.read_csv(EXPERIMENT_DIR / "training_history.csv")
history_df